# Dependency Installation

In [0]:
%pip install --upgrade pip
dbutils.library.restartPython()

%pip install "textacy==0.13.0" fastcoref huggingface_hub
dbutils.library.restartPython()

## Download Spacy English Model

In [0]:
dbutils.library.restartPython()
import spacy
spacy.prefer_gpu()
# Download English model
import spacy.cli; spacy.cli.download("en_core_web_sm")
dbutils.library.restartPython()

# Creating Spark Session

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.config("spark.sql.session.timeZone", "UTC").appName("FinSentAnalysis").getOrCreate()

# Defaults

In [0]:
WORKSPACE = "paid"
VOLUME = f'/Volumes/{WORKSPACE}/default/ensf612/'

# Reading News Data

In [0]:
news_df = spark.read.json(f"dbfs:{VOLUME}aapl_news.json").select(*['id', 'created', 'title', 'teaser', 'body'])

In [0]:
news_df.display()

# Curating Data

## Fixing Timestamps Types

In [0]:
news_df = news_df.withColumn('created', regexp_replace('created', r"^[A-Za-z]{3},\s+", "")).withColumn('created', to_timestamp('created', "dd MMM yyyy HH:mm:ss Z")).orderBy('created')
#price_df = price_df.withColumn('t', to_timestamp('t', "yyyy-MM-ddTHH:mm:ssZ")).orderBy('t')

## Text Preprocessing

1. Remove HTML Tags
2. Remove New Line, Tab, Carriage Return
3. Replace URL, Emails, Phone Numbers, Emojis, Hashtags, Social User Handles
4. Normalize Bullet Points, Quotation Marks, Multi Line Hyphenation, and White Spaces
5. Remove 'Image' and 'Also Read:..'

In [0]:
# Remove HTML Tags
from bs4 import BeautifulSoup as bs

def parse_html(text: str) -> str:
  return bs(text, 'html.parser').get_text()

# Remove New Line, Tab, Carriage Return
import re
def remove_carriage(text: str) -> str:
  return re.sub(r'\r|\n|\t', ' ', text)

# Remove 'Image' and 'Also Read'
def replace_irrelevant(text: str) -> str:
    return re.sub(r'Image:.*|Also Read: ', '', text)

# Create a Textacy pipeline
import networkx
from textacy.preprocessing import make_pipeline
from textacy.preprocessing.replace import emails, emojis, hashtags, phone_numbers, urls, user_handles
from textacy.preprocessing.normalize import bullet_points, quotation_marks, hyphenated_words, whitespace
text_pipe = make_pipeline(
    parse_html,
    remove_carriage,
    emails,
    emojis,
    hashtags,
    phone_numbers,
    urls,
    user_handles,
    bullet_points,
    quotation_marks,
    hyphenated_words,
    whitespace,
    replace_irrelevant
    )

# Convert into a Spark UDF
def text_preprocessing(text: str) -> str:
  return text_pipe(text)

In [0]:
pd_df = news_df.toPandas()
pd_df['title'] = pd_df['title'].apply(text_preprocessing)
pd_df['teaser'] = pd_df['teaser'].apply(text_preprocessing)
pd_df['body'] = pd_df['body'].apply(text_preprocessing)
news_df = spark.createDataFrame(pd_df)

In [0]:
news_df.display()

### Data Checkpoint

In [0]:
news_df.write.format("delta").mode("overwrite").saveAsTable("aapl_news_preprocessed")
#sp_price_df.write.json(f"dbfs:{VOLUME}aapl_price.json", mode="overwrite")

In [0]:
news_df = spark.read.table("aapl_news_preprocessed")

In [0]:
news_df.display()

## Coreference Resolution

**NOTE: We would have liked to do a bert based coreference before splitting the body of text into sentences but we keep getting into databricks serverless compute limitations. If we have classical clusters we would have ran this without any issue**

**Disabling this step for now**

### Local Model Cache

In [0]:
dbfs_local_tmp = f"{VOLUME}hf_models/fcoref"

In [0]:
model_tmp = ""
if 1:
    from huggingface_hub import snapshot_download
    import os
    import shutil

    local_tmp = "/tmp/hf_models/fcoref"
    os.makedirs(local_tmp, exist_ok=True)

    # Download HF snapshot into a normal local folder
    model_tmp = snapshot_download(
        "biu-nlp/f-coref",
        local_dir=local_tmp
    )

### Pandas UDF Function

In [0]:
if 1:
    from fastcoref import FCoref
    import pandas as pd
    import numpy as np

    HF_COREF_CACHE_DIR = str(model_tmp)

    _coref = None

    def get_coref():
        """
        Lazily initialize FCoref once per worker process.
        Called inside the pandas UDF.
        """
        global _coref
        if _coref is None:
            _coref = FCoref(
                model_name_or_path=HF_COREF_CACHE_DIR,
                device="cuda:0",              # serverless -> CPU
            )
        return _coref

    def get_resolved_text(result) -> str:

        if result is None:
            return None

        """
        Build a "resolved" text by replacing later mentions in each cluster
        with the first mention's surface string.
        """
        text = result.text
        clusters = result.get_clusters(as_strings=False)  # [[(start, end), ...], ...]

        # Collect replacements: (start, end, replacement_text)
        replacements = []

        for cluster in clusters:
            if not cluster:
                continue

            # First span is the canonical mention
            canonical_start, canonical_end = cluster[0]
            canonical_text = text[canonical_start:canonical_end]

            # Replace all *later* mentions with canonical text
            for (start, end) in cluster[1:]:
                replacements.append((start, end, canonical_text))

        # Sort by start index so we can rebuild left→right
        replacements.sort(key=lambda x: x[0])

        # Rebuild the text with replacements applied
        resolved_parts = []
        cur = 0

        for start, end, rep in replacements:
            # add text before this mention
            resolved_parts.append(text[cur:start])
            # add canonical form
            resolved_parts.append(rep)
            # move cursor
            cur = end

        # add the tail of the text
        resolved_parts.append(text[cur:])

        return "".join(resolved_parts)


    def coreference_resolution(col: pd.Series) -> pd.Series:
        """
        Spark pandas UDF:
        - col: pandas.Series of strings from a Spark column
        - returns: pandas.Series of resolved strings
        """
        mask = col.isna()

        texts = col.fillna("").tolist()
        n = len(texts)
        resolved_all = []

        coref = get_coref()          # lazy init per worker
        batch_size = 8               # tune if needed

        for start in range(0, n, batch_size):
            batch = texts[start:start + batch_size]
            if not batch:
                continue

            # predict(list[str]) -> list[CorefResult]
            preds = coref.predict(texts=batch)
            resolved_batch = [get_resolved_text(pred) for pred in preds]
            resolved_all.extend(resolved_batch)

        out = pd.Series(resolved_all, index=col.index)
        # restore original nulls
        out[mask] = None
        return out


In [0]:
pd_df = news_df.toPandas()
pd_df["body"] = coreference_resolution(pd_df["body"])
pd_df["title"] = coreference_resolution(pd_df["title"])
pd_df["teaser"] = coreference_resolution(pd_df["teaser"])
news_df = spark.createDataFrame(pd_df)

In [0]:
if 0:
  news_df = news_df.withColumn("body", coreference_resolution("body")).withColumn("title", coreference_resolution("title")).withColumn("teaser", coreference_resolution("teaser"))

  news_df.count()

In [0]:
if 0:
  news_df.display()

## Contextual Sentence Segmentation

### UDF Function

In [0]:
import en_core_web_sm

nlp = en_core_web_sm.load()

APPLE_NAMES = {
    "apple",
    "apple inc.",
    "apple, inc.",
    "apple incorporated",
}

def is_aapl_sentence(span):
    """
    Decide if a sentence is about Apple stock / company.
    Heuristics:
      - contains ticker 'AAPL'
      - or has ORG/PRODUCT entity with Apple name
    """
    text_lower = span.text.lower()

    # Check explicit ticker mention
    if "aapl" in text_lower or "apple" in text_lower:
        return True

    # Check NER entities
    for ent in span.ents:
        if ent.label_ in ("ORG", "PRODUCT"):
            if ent.text.lower() in APPLE_NAMES:
                return True

    return False

@udf
def split_sentences(text):
    doc = nlp(text)
    return '|'.join([sent.text.strip() for sent in doc.sents if is_aapl_sentence(sent)])

In [0]:
news_df = news_df.withColumn('body', split(split_sentences('body'), r'\|'))

news_df.count()

In [0]:
news_df.display()

# Silver Table

### Creation

In [0]:
news_df.write.format("delta").mode("overwrite").saveAsTable("aapl_news_curated")
#news_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news_curated.json", mode="overwrite")
#sp_price_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json", mode="overwrite")

### Reading

In [0]:
news_df = spark.read.table("aapl_news_curated")

# Sentiment Analysis

## Huggingface Pipeline

In [0]:
dbfs_local_tmp = f"{VOLUME}hf_models/finbert"

In [0]:
from huggingface_hub import snapshot_download
import os
import shutil

local_tmp = "/tmp/hf_models/finbert"
os.makedirs(local_tmp, exist_ok=True)

# Download HF snapshot into a normal local folder
model_tmp = snapshot_download(
    "ProsusAI/finbert",
    local_dir=local_tmp
)

shutil.copytree(model_tmp, dbfs_local_tmp, dirs_exist_ok=True)

In [0]:
import os

HF_CACHE_DIR = f"/tmp"

os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR
os.environ["HF_HOME"] = HF_CACHE_DIR

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

_classifier = None

def get_classifier():
    """
    Lazily initialize the FinBERT pipeline on each worker.
    This function runs ON THE EXECUTOR, not the client,
    so the model is not serialized over gRPC.
    """
    global _classifier
    if _classifier is None:
        _classifier = pipeline("text-classification", model=dbfs_local_tmp,
    tokenizer=dbfs_local_tmp, top_k=1, device=device)
    return _classifier

## Pandas UDF

In [0]:
import pandas as pd
from collections import Counter
#from tqdm.auto import tqdm

@pandas_udf(ArrayType(MapType(StringType(), DoubleType())))
def classify_text_udf(bodies: pd.Series) -> pd.Series:
    """
    bodies: each row is a Python list of sentences (Spark array<string>)

    returns: each row is a list of maps:
      [
        {"positive": 0.1},
        {"positive": 0.6},
        ...
      ]
    """
    out = []
    clf = get_classifier()  # model created on worker the first time

    for sentences in bodies:
        # handle nulls
        if (
        sentences is None
        or (isinstance(sentences, str) and sentences.strip() == "")
        ):
            out.append(None)
            continue

        if isinstance(sentences, str):
            sentences = [sentences]
        
        try:
            # run FinBERT on the list of sentences
            preds = clf(
                sentences
            )
        except:
            out.append(None)
            continue
        
        top_labels = []
        top_label_scores = []  # list of (label, score)
        for per_sentence in preds:
            # per_sentence is a list like:
            # [{"label": "positive", "score": ...}, {"label": "negative", ...}, ...]
            (label, score) = {item["label"]: float(item["score"]) for item in per_sentence}.items()
            top_labels.append(label)
            top_label_scores.append((label, score))

        # --- 3. combine per-sentence sentiments ---
        # most frequent top label
        label_counts = Counter(top_labels)
        combined_label = label_counts.most_common(1)[0][0]

        # mean score for that label across sentences where it was top
        scores_for_label = [score for lbl, score in top_label_scores if lbl == combined_label]
        combined_score = sum(scores_for_label) * 1.0 / len(scores_for_label)

        # final single-label map
        combined_sentiment = {combined_label: combined_score}
        out.append(combined_sentiment)

    return pd.Series(out)

In [0]:
news_df_trunc = news_df.withColumn('sentiment_body', classify_text_udf('body')).withColumn('sentiment_title', classify_text_udf('title')).withColumn('sentiment_teaser', classify_text_udf('teaser'))

In [0]:
news_df_trunc.display()

In [0]:
news_df_trunc.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("aapl_sentiment")

## Sentiment Collection

In [0]:
# Average based on date
# Average based on title, teaser, body

# Feature Engineering

In [0]:
# combine with daily price returns and shift one day to capture the returns for next day based on market data

# Gold Table

In [0]:
news_df_trunc.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("aapl_sentiment")

# Resources

1. [https://arxiv.org/pdf/2306.02136](https://arxiv.org/pdf/2306.02136)